# 00 - Data preparation

Builds every shared input for the rest of the pipeline. Run once; all other notebooks attach its output.

| Output | Content |
|---|---|
| `png1024/` | one 1024x1024 PNG per image (train + test) |
| `meta.csv` | image_id, split, rows, cols (original size, needed to map boxes back) |
| `targets_fused.csv` | fused boxes per class: image_id, class_id, x_min..y_max (0-1), n_rads |
| `cls_labels.csv` | abnormal flag + 14 soft class labels (fraction of radiologists) |
| `folds.csv` | 5 multilabel-stratified folds, shared by every model |
| `det_train_fold{k}.txt` | detector training list per fold: abnormal images + 25% of normal images |

**Runtime:** CPU notebook is enough. The DICOM conversion is the slow part (roughly 1-2 hours). Output size is roughly 10 GB, within Kaggle's working-directory limit.

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'

In [ ]:
!pip install -q ensemble-boxes iterative-stratification
import pydicom, cv2
from joblib import Parallel, delayed
from tqdm.auto import tqdm
try:
    from pydicom.pixels import apply_voi_lut          # pydicom >= 3
except ImportError:
    from pydicom.pixel_data_handlers.util import apply_voi_lut

DATA_DIR = U.competition_dir()
PNG_DIR = f'{OUT}/png1024'
os.makedirs(PNG_DIR, exist_ok=True)
N_FOLDS, SEED, NORMAL_FRAC = 5, 42, 0.25
print(DATA_DIR)

## 1. DICOM -> PNG
VOI LUT (scanner window), MONOCHROME1 inversion (~18% of images), percentile normalization (robust to the 3.6% of files without a window), resize to 1024x1024.

In [ ]:
def dicom_to_png(path, split):
    image_id = os.path.basename(path)[:-6]
    d = pydicom.dcmread(path)
    rows, cols = int(d.Rows), int(d.Columns)
    out = f'{PNG_DIR}/{image_id}.png'
    if not os.path.exists(out):                                    # resumable
        img = apply_voi_lut(d.pixel_array, d).astype(np.float32)
        if d.PhotometricInterpretation == 'MONOCHROME1':
            img = img.max() - img                                  # invert so bones are white
        lo, hi = np.percentile(img, (0.5, 99.5))
        img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
        img = cv2.resize((img * 255).astype(np.uint8), (U.IMG_SIZE, U.IMG_SIZE), interpolation=cv2.INTER_AREA)
        cv2.imwrite(out, img)
    return dict(image_id=image_id, split=split, rows=rows, cols=cols,
                photometric=d.PhotometricInterpretation)

paths = [(p, 'train') for p in glob.glob(f'{DATA_DIR}/train/*.dicom')] + \
        [(p, 'test') for p in glob.glob(f'{DATA_DIR}/test/*.dicom')]
meta = pd.DataFrame(Parallel(n_jobs=-1)(delayed(dicom_to_png)(p, s) for p, s in tqdm(paths)))
meta.to_csv(f'{OUT}/meta.csv', index=False)
print(meta.split.value_counts().to_dict(), '| PNG files:', len(os.listdir(PNG_DIR)))

In [ ]:
# Visual check: random images incl. MONOCHROME1 - bones must be white in all of them
pick = pd.concat([meta[meta.photometric == 'MONOCHROME1'].sample(4, random_state=0),
                  meta[meta.photometric == 'MONOCHROME2'].sample(4, random_state=0)])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.ravel(), pick.itertuples()):
    ax.imshow(cv2.imread(f'{PNG_DIR}/{r.image_id}.png', 0), cmap='gray'); ax.axis('off')
    ax.set_title(f'{r.photometric} | {r.split}', fontsize=9)
plt.tight_layout(); plt.show()

## 2. Clean and normalize the boxes
Drop the 3 degenerate boxes (shorter side < 10 px, all from R16). Normalize coordinates by each image's original size.

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
abn = train[train.class_id != U.NO_FINDING].copy()
short = np.minimum(abn.x_max - abn.x_min, abn.y_max - abn.y_min)
print('degenerate boxes removed:', int((short < 10).sum()))
abn = abn[short >= 10].merge(meta[['image_id', 'rows', 'cols']], on='image_id')
abn['nx_min'], abn['nx_max'] = abn.x_min / abn.cols, abn.x_max / abn.cols
abn['ny_min'], abn['ny_max'] = abn.y_min / abn.rows, abn.y_max / abn.rows
NBOX = ['nx_min', 'ny_min', 'nx_max', 'ny_max']
print('boxes:', len(abn), '| abnormal images:', abn.image_id.nunique())

## 3. Fused targets
Each radiologist is treated as one "model" in WBF (score 1 for every box). WBF only merges boxes of the same class, so fusion is per class. **No box is dropped**: boxes marked by a single radiologist stay as targets. For every fused box we keep `n_rads`, the number of radiologists whose box overlaps it (IoU >= 0.4).

In [ ]:
from ensemble_boxes import weighted_boxes_fusion
IOU_THR = 0.4

def fuse_image(g):
    boxes, scores, labels = [], [], []
    for _, r in g.groupby('rad_id'):
        boxes.append(r[NBOX].values.clip(0, 1).tolist()); scores.append([1.0] * len(r)); labels.append(r.class_id.tolist())
    fb, _, fl = weighted_boxes_fusion(boxes, scores, labels, iou_thr=IOU_THR, skip_box_thr=0.0)
    out = []
    for b, l in zip(fb, fl):
        same = g[g.class_id == l]
        ious = U.iou_matrix(b, same[NBOX].values)[0]
        out.append(dict(image_id=g.image_id.iloc[0], class_id=int(l), x_min=b[0], y_min=b[1], x_max=b[2], y_max=b[3],
                        n_rads=max(1, same.rad_id[ious >= IOU_THR].nunique())))
    return out

res = Parallel(n_jobs=-1)(delayed(fuse_image)(g) for _, g in tqdm(abn.groupby('image_id')))
targets = pd.DataFrame([r for rr in res for r in rr])
targets.to_csv(f'{OUT}/targets_fused.csv', index=False)

print('fused boxes:', len(targets), 'from', len(abn), 'radiologist boxes | images:', targets.image_id.nunique())
display(pd.crosstab(targets.class_id.map(dict(enumerate(U.CLASSES))), targets.n_rads, margins=True))

In [ ]:
# Visual check of the fusion: radiologist boxes (thin, colored) vs fused box (thick, white)
import matplotlib.patches as patches
ids = targets.image_id.drop_duplicates().sample(4, random_state=1)
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, i in zip(axes, ids):
    ax.imshow(cv2.imread(f'{PNG_DIR}/{i}.png', 0), cmap='gray'); ax.axis('off')
    for r, col in zip(sorted(abn[abn.image_id == i].rad_id.unique()), ['red', 'cyan', 'yellow']):
        for b in abn[(abn.image_id == i) & (abn.rad_id == r)][NBOX].values * U.IMG_SIZE:
            ax.add_patch(patches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec=col, lw=0.8))
    for t in targets[targets.image_id == i].itertuples():
        b = np.array([t.x_min, t.y_min, t.x_max, t.y_max]) * U.IMG_SIZE
        ax.add_patch(patches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec='white', lw=2))
        ax.text(b[0], b[1] - 5, f'{U.CLASSES[t.class_id][:12]} n={t.n_rads}', color='white', fontsize=7)
plt.tight_layout(); plt.show()

## 4. Classifier labels
`abnormal`: 1 if any radiologist drew a box. `c0..c13`: fraction of radiologists who marked the class in the image, in {0, 1/3, 2/3, 1}.

In [ ]:
train_ids = meta.loc[meta.split == 'train', 'image_id']
votes = abn.groupby(['image_id', 'class_id']).rad_id.nunique().unstack(fill_value=0)
soft = (votes / 3).reindex(index=train_ids, columns=range(14), fill_value=0)
cls_labels = soft.add_prefix('c')
cls_labels.insert(0, 'abnormal', (soft.sum(1) > 0).astype(int))
cls_labels.index.name = 'image_id'
cls_labels.to_csv(f'{OUT}/cls_labels.csv')
print(len(cls_labels), 'images | abnormal:', cls_labels.abnormal.sum())
print('soft label values:', sorted(np.unique(soft.values.round(3))))

## 5. Folds
One split for every model - the condition for fusing out-of-fold predictions without leakage.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
Y = np.c_[(soft.values > 0), cls_labels.abnormal.values]
folds = pd.Series(-1, index=soft.index, name='fold')
for k, (_, va) in enumerate(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(Y)), Y)):
    folds.iloc[va] = k
folds.to_csv(f'{OUT}/folds.csv')

check = pd.DataFrame({'images': folds.value_counts().sort_index(),
                      'abnormal_rate': cls_labels.abnormal.groupby(folds).mean().round(3)})
per_class = (soft > 0).groupby(folds).sum(); per_class.columns = U.CLASSES
display(check); display(per_class)

## 6. Detector training lists
All abnormal images of the training folds + 25% of their normal images (negatives). Both detectors use exactly the same lists.

In [ ]:
abn_set = set(targets.image_id)
rng = np.random.RandomState(SEED)
for k in range(N_FOLDS):
    tr = folds.index[folds != k]
    normals = [i for i in tr if i not in abn_set]
    n_norm = int(len(normals) * NORMAL_FRAC)
    use = [i for i in tr if i in abn_set] + list(rng.choice(normals, n_norm, replace=False))
    with open(f'{OUT}/det_train_fold{k}.txt', 'w') as f:
        f.write('\n'.join(use))
    print(f'fold {k}: {len(use)} training images ({len(use) - n_norm} abnormal + {n_norm} normal)')

## 7. Evaluator sanity check
Feeding the ground truth as predictions must give mAP = 1.0. If not, no number downstream can be trusted.

In [ ]:
gt = U.build_gt(targets, folds.index[folds == 0])
r = U.evaluate(gt, gt.assign(score=1.0)[U.PRED_COLS])
assert abs(r['map15'] - 1.0) < 1e-6, 'evaluator is broken'
print('Evaluator OK')

In [ ]:
for f in ['meta.csv', 'targets_fused.csv', 'cls_labels.csv', 'folds.csv']:
    print(f, os.path.getsize(f'{OUT}/{f}') // 1024, 'KB')